# Evaluate fine-tuned models on the BBCH image–text benchmark (step 2)

Loads the benchmark built by `31.build_benchmark.ipynb` (`benchmark/benchmark.csv` + `benchmark/stage_lists.csv`, cached in `vars/`) and scores **image → text** retrieval with the best-performing fine-tuned image/text models.

## Retrieval tasks (image → text only)

1. **image → crop** — the candidate pool is the crop-name list from `cropnames.txt`. One ranking per photo.
2. **within-crop image → stage** — the candidate pool is restricted to the photo's own crop, and the two levels are evaluated **independently**:
   - **principal** stage (coarse BBCH digit / Terres Inovia letter)
   - **secondary** stage (2-digit BBCH / Terres Inovia code)

Both tasks are reported **separately for French and English** descriptions, never mixed.

## Metrics

R@1, R@3, R@5, R@10 and MRR — overall and broken down per crop. Same rank computation as the GT evaluation protocol used throughout `Finetuning/`:

```python
ranks = (sim_matrix >= sim_matrix.diag().unsqueeze(1)).sum(dim=1)
```

## Models evaluated

- `00_siglip2_out_of_box` — pretrained SigLIP2, no heads (out-of-box reference; chunk + max-over-chunks text)
- **`smaller_size_*`** — the notebook-28 `TextImage_Smaller_train_sizes_runs` sweep: DINOv3 + BGE-M3 (last 6 layers unfrozen) trained on 100 / 500 / 1000 / 2500 / 5000 / 10000 / 20000 / 33526 pairs. This **replaces run 06**, which is the same architecture at one training size.
- `29_sweep_size_*` — the notebook-29 in-batch-mining size sweep (DINOv3 + BGE-M3, last 6 layers unfrozen)
- `27_last{n}_layers` — notebook-27 unfrozen-layer sweep (DINOv3 + BGE-M3, `last_{0,2,4,6,8}_layers`)
- `27C_last{n}_layers` — notebook-27C SigLIP2 unfrozen-layer sweep (max-over-chunks text)
- `27D_last{n}_layers` — notebook-27D SigLIP2 unfrozen-layer sweep (**mean-pooled** text, matching its training)
- `26_zs_siglip2_trunc` / `26_zs_siglip2_meanpool` — notebook-26 zero-shot SigLIP2 baselines (truncate at 64 tokens / chunk + mean-pool)
- `26_zs_clip_meanpool` — notebook-26 zero-shot CLIP baseline (chunk + mean-pool); its truncate variant is identical to `zz_clip_raw`
- `zz_random` — random-embedding control (metric sanity check)
- `zz_dinov3_bge_raw` — pretrained DINOv3 + BGE-M3, no trained heads (no-training floor)
- `zz_clip_raw` — pretrained CLIP ViT-B/32, no heads (E7: second out-of-box family; also NB26's `clip_truncate77` zero-shot baseline)



## Intermediate artifacts

Everything reusable is cached to the repo-root `vars/` with the same `pkl_vars` / `reload_vars` helpers as notebook 29, so re-running the notebook skips re-encoding. Models flagged `recompute` in the config (currently `smaller_size_*`, whose ids now point at the NB28 runs) always re-encode and overwrite their cache.


In [ ]:
# Resolve paths from the repository root so notebooks work regardless of the kernel CWD.
from pathlib import Path
import os

def _find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "manifests" / "corpus_manifest.csv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the PhytoPairs repository root (expected data/manifests/corpus_manifest.csv)."
    )

PROJECT_ROOT = _find_project_root()
BSV_ROOT = PROJECT_ROOT / "3.Dataset Construction"
FINETUNING_ROOT = PROJECT_ROOT / "4.Finetuning"
CASE_STUDY_ROOT = PROJECT_ROOT / "5.BBCH case study"
RESULTS_RUNS_ROOT = PROJECT_ROOT / "results" / "runs" / "section_4"
DEFAULT_HF_CACHE = os.environ.get(
    "HF_HOME", str(Path.home() / ".cache" / "huggingface")
)


## Section 1 — Setup

In [ ]:
import json
import os
import pickle
import re
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer, AutoProcessor
from transformers import CLIPModel, CLIPProcessor

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
#from huggingface_hub import login
#login()

## Section 2 — Config

Paths, the Hugging Face cache, and the **configurable model scope**.

- `HF_CACHE` — one constant, resolved from a list of candidate locations and passed explicitly to every `from_pretrained()`. The ambient `HF_HOME` env var is unreliable here, so it is never relied on alone. `CLIP_CACHE` is separate because `openai/clip-vit-base-patch32` lives in `hf_cache`, not `hf_cache2`.
- `MODELS` — the list of single-run models to evaluate. Each entry is `{"id", "loader", "dir", "files"?, "text_mode"?, "text_pooling"?, "recompute"?}`; nothing downstream hard-codes a checkpoint path or model id. Loader families (`dinov3_bge`, `siglip2_bge`, `siglip2_raw`, `dinov3_bge_raw`, `clip_raw`, `random`) are generic, so a new checkpoint plugs in without code changes. `text_mode` / `text_pooling` pick the text reduction a model was trained with; `recompute` marks ids whose embedding cache is stale by construction.
- `SWEEPS` — one entry per swept family: the notebook-27/27C/27D unfrozen-layer sweeps (`last_{0,2,4,6,8}_layers`), the notebook-28 `smaller_size_*` size sweep (replaces run 06), and the notebook-29 in-batch-mining size sweep. Each entry is `{"id_prefix", "root", "values", "dir_template"?, "loader", "files"?, "text_pooling"?, "recompute"?}`. Values whose run folder holds no weights are skipped automatically.
- `RECOMPUTE` — built from the `recompute` flags; those ids always re-encode and overwrite their `vars/bbch_emb_<id>.pkl` cache (currently `smaller_size_*`, whose ids now point at the NB28 runs).

The registry builder (Section 5) registers only the entries whose checkpoint folder exists and prints what was **registered** vs. **skipped**, with the reason.


In [17]:
# --- paths -----------------------------------------------------------------
ROOT = CASE_STUDY_ROOT
PHOTOS_DIR = ROOT / "photos"
BENCH_DIR = ROOT / "benchmark"
VARS_DIR = CASE_STUDY_ROOT / "vars"
RESULTS_DIR = ROOT / "results"
PREDICTION_TOP_K = 5

BENCHMARK_CSV = BENCH_DIR / "benchmark.csv"
STAGE_LISTS_CSV = BENCH_DIR / "stage_lists.csv"

# --- Hugging Face cache ----------------------------------------------------
# One constant, passed explicitly to every from_pretrained() call. The ambient HF_HOME
# env var has proven unreliable here, so never rely on it alone.
# This must be a FLAT hub cache (it directly contains models--<org>--<name>/ folders);
# do not point it at an HF_HOME root that has a hub/ subfolder.

HF_CACHE =  DEFAULT_HF_CACHE
os.environ.setdefault("HF_HOME", HF_CACHE)

# --- metrics ---------------------------------------------------------------
K_VALUES = [1, 3]
METRIC_COLS = [f"R@{k}" for k in K_VALUES] + ["MRR"]

# --- encoders --------------------------------------------------------------
DINOV3_MODEL_NAME = "facebook/dinov3-vitb16-pretrain-lvd1689m"
BGE_MODEL_NAME = "BAAI/bge-m3"
SIGLIP2_MODEL_NAME = "google/siglip2-base-patch16-224"

# E7 -- second out-of-box family. NOTE: openai/clip-vit-base-patch32 lives in the OTHER
# cache (hf_cache, not hf_cache2), so it gets its own constant.
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
CLIP_CACHE = DEFAULT_HF_CACHE
CLIP_MAX_LENGTH = 77          # CLIP's text tower hard limit

SHARED_DIM = 512
BGE_MAX_LENGTH = 512          # stage descriptions are short; fixed for all models
SIGLIP2_TEXT_CHUNK_TOKENS = 64
IMAGE_ENCODE_CHUNK_SIZE = 16

# --- checkpoints -----------------------------------------------------------
# All trained runs live under results/runs/section_4 -- the NB 26-29B output root
# (baselines, layer_sweeps, size_sweeps, noise_sweeps).
# The old yabusaqe/Finetuning tree is RETIRED: its old-pipeline runs (02, the
# pre-NB28 Smaller_train_sizes sweep) are no longer evaluated.
CROSS_ENCODER_FT = Path(str(RESULTS_RUNS_ROOT))

# Notebook-29 in-batch-mining sweep (NB 29's own OUTPUT_DIR).
CHECKPOINT_ROOT = CROSS_ENCODER_FT / "noise_sweeps" / "textimage_inbatch_mining_sweep"

# --- model scope (CONFIGURABLE) --------------------------------------------
# Populate this list with the models to evaluate. Nothing below hard-codes a checkpoint
# path or a model id: the registry builder iterates MODELS, registers the entries whose
# checkpoint folder exists, and prints what was registered vs. skipped (with the reason).
#
# Each entry is a dict:
#   id        : unique model id; also the embedding cache name (vars/bbch_emb_<id>.pkl)
#               and the results file name (benchmark/bbch_results_<id>.csv)
#   loader    : one of "dinov3_bge" | "siglip2_bge" | "siglip2_raw" | "dinov3_bge_raw"
#               | "clip_raw" | "random"
#   dir       : checkpoint folder (None for the *_raw / "random" loaders, which need no weights)
#   files     : optional dict overriding the checkpoint file names, e.g.
#               {"head_image": "head_image_dinov3.pt", "head_text": "head_text_bge.pt"}
#   text_mode : (raw loaders only) how long texts are turned into vectors:
#               "truncate"    -- one vector from the first native-context tokens (NB26's
#                                literature-standard zero-shot)
#               "chunks_mean" -- chunk at the native context, encode, mean-pool + renorm
#                                (NB26's mean-pool baselines)
#               "chunks_max"  -- chunk and keep the chunks as a multi-vector (default)
#   text_pooling : ("siglip2_bge" only) "max" (default, multi-vector) or "mean"
#               (mean-pool the projected chunks + renorm, single vector -- NB27D's training)
#   recompute : if True, always re-encode and overwrite vars/bbch_emb_<id>.pkl. Use when
#               an id's meaning changed (e.g. smaller_size_* now points at the NB28 runs),
#               otherwise the stale cache would silently win.
#
# Loader families stay generic, so a new checkpoint plugs in without code changes.
MODELS = [
    {
        "id": "00_siglip2_out_of_box",
        "loader": "siglip2_raw",
        "dir": None,          # pretrained SigLIP2, no projection heads
    },
    # Old-pipeline runs are out of scope: run 06 is one point of the smaller_size_* curve
    # below, and run 02 (SigLIP2 + heads, last-2 layers unfrozen) is superseded by
    # 27C_last_2 -- the same recipe retrained with the FN-masked pipeline. The
    # 26_zs_siglip2_* rows below are the zero-shot baselines for that family.
    # NB26 zero-shot baselines. SigLIP2 is scored in both of NB26's text modes (truncate at
    # the native context, or chunk + mean-pool). CLIP's truncate mode is identical to
    # zz_clip_raw below (same weights, same 77-token truncation), so only its mean-pool
    # variant is added here; zz_clip_raw doubles as NB26's clip_truncate77 row.
    {
        "id": "26_zs_siglip2_trunc",
        "loader": "siglip2_raw",
        "dir": None,
        "text_mode": "truncate",
    },
    {
        "id": "26_zs_siglip2_meanpool",
        "loader": "siglip2_raw",
        "dir": None,
        "text_mode": "chunks_mean",
    },
    {
        "id": "26_zs_clip_meanpool",
        "loader": "clip_raw",
        "dir": None,
        "text_mode": "chunks_mean",
    },
    # Control: random embeddings, no model. Its scores must land near the analytic chance
    # level -- if they do not, the metric code is wrong. Cheap (no model load).
    {
        "id": "zz_random",
        "loader": "random",
        "dir": None,
    },
    # Baseline: pretrained DINOv3 + BGE-M3, NO trained heads (fixed random projection into
    # the shared space). Completes the 2x2 out-of-box vs fine-tuned x SigLIP2 vs DINOv3+BGE.
    # Expect near-chance scores: the two encoders are not natively aligned.
    {
        "id": "zz_dinov3_bge_raw",
        "loader": "dinov3_bge_raw",
        "dir": None,
    },
    # E7: second out-of-box family. CLIP's two towers ARE natively aligned (contrastively
    # pretrained together), so this needs no projection and no training -- it tests whether
    # SigLIP2's out-of-box lead is model-specific or a property of aligned pairs in general.
    # Also NB26's clip_truncate77 zero-shot baseline (identical embeddings).
    {
        "id": "zz_clip_raw",
        "loader": "clip_raw",
        "dir": None,
    },
]

# --- sweeps (one configurable entry per family) -----------------------------
# Each entry registers one model per value: <id_prefix>_<value>. A value is registered
# only when its folder holds weights (the full checkpoint OR both head files); missing
# values are reported as skipped, so a partial sweep still runs.
#
#   id_prefix   : model-id prefix (also the embedding cache / results-file name)
#   root        : parent folder of the per-value run folders
#   values      : the swept values (training sizes, or n_unfrozen_layers)
#   dir_template: run-folder name template ("size_{value}" or "last_{value}_layers")
#   loader      : loader family ("dinov3_bge" | "siglip2_bge")
#   files       : checkpoint file names (see load_dinov3_bge / load_siglip2_bge)
#   text_pooling: "max" | "mean" (siglip2_bge only)
#   recompute   : force re-encoding (see MODELS above)
SWEEPS = [
    # Notebook-29 in-batch-mining size sweep (DINOv3 + BGE-M3, last 6 layers unfrozen).
    # Sizes whose weights are absent are skipped and re-registered automatically once
    # the training run writes them.
    {
        "id_prefix": "29_sweep_size",
        "root": CHECKPOINT_ROOT,
        "values": [100, 500, 1000, 2500, 5000, 10000, 20000, 33208],
        "dir_template": "size_{value}",
        "loader": "dinov3_bge",
        "files": {"head_image": "head_image_dinov3.pt", "head_text": "head_text_bge.pt"},
    },
    # Notebook-28 TextImage size sweep (DINOv3 + BGE-M3, last 6 layers unfrozen), trained
    # with the in-batch FN-masked pipeline. Same architecture and size points as run 06,
    # so this replaces it. Every size keeps full artifacts (heads + tuned encoders +
    # latest_checkpoint.pt), unlike the retired yabusaqe/Finetuning/Smaller_train_sizes
    # runs whose smallest sizes kept heads only. `latest_checkpoint.pt` holds EVERY tower
    # plus the epoch and is preferred whenever present; the individual-file names remain
    # as a fallback. The ids keep the historical `smaller_size_*` names, so `recompute`
    # is set: the old caches were built from the retired runs and must not win.
    {
        "id_prefix": "smaller_size",
        "root": CROSS_ENCODER_FT / "Smaller_train_sizes" / "TextImage_Smaller_train_sizes_runs",
        "values": [100, 500, 1000, 2500, 5000, 10000, 20000, 33526],
        "dir_template": "size_{value}",
        "loader": "dinov3_bge",
        "files": {
            "checkpoint": "latest_checkpoint.pt",
            "head_image": "head_image_dinov3.pt",
            "head_text": "head_text_bge.pt",
            "encoder_image": ["dinov3_model_unfrozen.pt", "dinov3_model.pt"],
            "encoder_text": ["bge_model_unfrozen.pt", "bge_model.pt"],
        },
        "recompute": False,
    },
    # Notebook-27 unfrozen-layer sweep (DINOv3 + BGE-M3): the swept variable is the number
    # of unfrozen LAST layers in both encoders, not the training size. Values whose folder
    # is missing (e.g. last_8_layers before it is trained) are skipped automatically.
    {
        "id_prefix": "27_last",
        "root": CROSS_ENCODER_FT / "diff_unfrozen_layers" / "textimage_fn_masked",
        "values": [0, 2, 4, 6, 8],
        "dir_template": "last_{value}_layers",
        "loader": "dinov3_bge",
        "files": {
            "checkpoint": "latest_checkpoint.pt",
            "head_image": "head_image_dinov3.pt",
            "head_text": "head_text_bge.pt",
            "encoder_image": ["dinov3_model_unfrozen.pt", "dinov3_model.pt"],
            "encoder_text": ["bge_model_unfrozen.pt", "bge_model.pt"],
        },
    },
    # Notebook-27C: same layer sweep with SigLIP2 for BOTH towers (image + 64-token chunked
    # text). Text stays multi-vector (max over chunks), matching its training.
    {
        "id_prefix": "27C_last",
        "root": CROSS_ENCODER_FT / "diff_unfrozen_layers" / "textimage_siglip2_fn_masked",
        "values": [0, 2, 4, 6, 8],
        "dir_template": "last_{value}_layers",
        "loader": "siglip2_bge",
        "files": {
            "checkpoint": "latest_checkpoint.pt",
            "head_image": "head_image_siglip2.pt",
            "head_text": "head_text_siglip2.pt",
            "encoder_image": "siglip2_vision_unfrozen.pt",
            "encoder_text": "siglip2_text_unfrozen.pt",
        },
    },
    # Notebook-27D: NB27C with mean-pooled text (mean over projected chunks + renorm), the
    # reduction it was trained and GT-evaluated with. text_pooling="mean" makes the loader
    # score it the same way.
    {
        "id_prefix": "27D_last",
        "root": CROSS_ENCODER_FT / "diff_unfrozen_layers" / "textimage_siglip2_meanpool",
        "values": [0, 2, 4, 6, 8],
        "dir_template": "last_{value}_layers",
        "loader": "siglip2_bge",
        "files": {
            "checkpoint": "latest_checkpoint.pt",
            "head_image": "head_image_siglip2.pt",
            "head_text": "head_text_siglip2.pt",
            "encoder_image": "siglip2_vision_unfrozen.pt",
            "encoder_text": "siglip2_text_unfrozen.pt",
        },
        "text_pooling": "mean",
    },
]

# --- stale-cache guard ------------------------------------------------------
# Ids whose cached embeddings are stale by construction (their config changed since the
# cache was written, e.g. smaller_size_* now points at the NB28 runs). encode_model()
# always re-encodes these and overwrites vars/bbch_emb_<id>.pkl.
RECOMPUTE = {spec["id"] for spec in MODELS if spec.get("recompute")}
for sweep in SWEEPS:
    if sweep.get("recompute"):
        RECOMPUTE.update(f"{sweep['id_prefix']}_{value}" for value in sweep["values"])
if RECOMPUTE:
    print("RECOMPUTE (cache ignored):", sorted(RECOMPUTE))

print("ROOT:", ROOT)
print("VARS_DIR:", VARS_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("HF_CACHE:", HF_CACHE, "(exists:", Path(HF_CACHE).exists(), ")")
print("CLIP_CACHE:", CLIP_CACHE, "(exists:", Path(CLIP_CACHE).exists(), ")")
print("CROSS_ENCODER_FT exists:", CROSS_ENCODER_FT.exists())
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT, "(exists:", CHECKPOINT_ROOT.exists(), ")")
print(f"MODELS configured: {len(MODELS)}")
for spec in MODELS:
    d = spec["dir"]
    exists = "n/a (pretrained)" if d is None else d.exists()
    print(f"  {spec['id']}: {d} -> exists: {exists}")
for sweep in SWEEPS:
    print(f"SWEEP {sweep['id_prefix']}: {sweep['root']} "
          f"values={sweep['values']} (exists: {Path(sweep['root']).exists()})")

def pkl_vars(varname, filename):
    """Pickle a variable into VARS_DIR (same helper as notebook 29)."""
    with open(VARS_DIR / filename, "wb") as file:
        pickle.dump(varname, file)


def reload_vars(filename):
    """Load a variable previously written by pkl_vars()."""
    with open(VARS_DIR / filename, "rb") as file:
        return pickle.load(file)


def cache_exists(filename):
    return (VARS_DIR / filename).exists()


print("vars/ helpers ready.")


ROOT: /projects/agrifutur/sji/bsv-crossmodalIR/BBCH case study
VARS_DIR: /projects/agrifutur/sji/bsv-crossmodalIR/vars
RESULTS_DIR: /projects/agrifutur/sji/bsv-crossmodalIR/BBCH case study/results
HF_CACHE: /projects/agrifutur/hf_cache2 (exists: True )
CLIP_CACHE: /projects/agrifutur/hf_cache (exists: True )
CROSS_ENCODER_FT exists: True
CHECKPOINT_ROOT: /projects/agrifutur/sji/cross_encoder_ft/mined_negatives_ft/textimage_inbatch_mining_sweep (exists: True )
MODELS configured: 7
  00_siglip2_out_of_box: None -> exists: n/a (pretrained)
  26_zs_siglip2_trunc: None -> exists: n/a (pretrained)
  26_zs_siglip2_meanpool: None -> exists: n/a (pretrained)
  26_zs_clip_meanpool: None -> exists: n/a (pretrained)
  zz_random: None -> exists: n/a (pretrained)
  zz_dinov3_bge_raw: None -> exists: n/a (pretrained)
  zz_clip_raw: None -> exists: n/a (pretrained)
SWEEP 29_sweep_size: /projects/agrifutur/sji/cross_encoder_ft/mined_negatives_ft/textimage_inbatch_mining_sweep values=[100, 500, 1000, 25

## Section 3 — Load the benchmark

Reads `df_bench` (one row per photo) and the per-crop stage lists. Prefers the `vars/` pickles written by notebook 31 and falls back to the CSVs if they are missing.

In [4]:
if cache_exists("df_bench.pkl"):
    df_bench = reload_vars("df_bench.pkl")
    print("Loaded df_bench from vars/df_bench.pkl")
else:
    df_bench = pd.read_csv(BENCHMARK_CSV, encoding="utf-8")
    print(f"Loaded df_bench from {BENCHMARK_CSV}")

if cache_exists("bbch_stage_lists.pkl"):
    df_stage_lists = reload_vars("bbch_stage_lists.pkl")
    stage_lists = reload_vars("bbch_stage_lists_dict.pkl")
    crop_candidates = reload_vars("bbch_crop_candidates.pkl")
    print("Loaded stage lists from vars/")
else:
    df_stage_lists = pd.read_csv(STAGE_LISTS_CSV, encoding="utf-8")
    stage_lists = defaultdict(list)
    for row in df_stage_lists.to_dict("records"):
        stage_lists[(row["crop_name"], row["level"])].append(row)
    stage_lists = dict(stage_lists)
    crop_candidates = sorted(df_bench["crop_name"].unique())
    print(f"Loaded stage lists from {STAGE_LISTS_CSV}")

print(f"\nBenchmark photos: {len(df_bench)}")
print(f"Crops: {sorted(df_bench['crop_name'].unique())}")
print(f"Crop candidates ({len(crop_candidates)}): {crop_candidates}")
print(f"Stage-list rows: {len(df_stage_lists)}")
print(df_stage_lists.groupby(['crop_name', 'level']).size().to_string())

Loaded df_bench from vars/df_bench.pkl
Loaded stage lists from vars/

Benchmark photos: 94
Crops: ['citrus', 'oilseedFlax', 'oilseedRape']
Crop candidates (32): ['bean', 'beet', 'bulbVegetables', 'cellfie', 'cereals', 'chickpea', 'citrus', 'cucurbits', 'currants', 'fabaBean', 'grapevine', 'hemp', 'leafVegetablesFormingHeads', 'lentil', 'lupin', 'maize', 'oilseedFlax', 'oilseedRape', 'oliveTree', 'otherBrassicaVegetables', 'pea', 'photos', 'pomeFruits', 'potato', 'rice', 'rootAndStemVegetables', 'solanaceousFruits', 'soybean', 'stage_excels', 'stoneFruits', 'strawberry', 'sunflower']
Stage-list rows: 161
crop_name    level    
citrus       principal     8
             secondary    35
oilseedFlax  principal     9
             secondary    81
oilseedRape  secondary    28


In [5]:
# Resolve each photo's absolute path and its ground-truth ids, once.
df_bench["image_path"] = df_bench["photo_name"].apply(lambda n: str(PHOTOS_DIR / n))

missing_images = [p for p in df_bench["image_path"] if not os.path.exists(p)]
assert not missing_images, f"missing image files: {missing_images[:5]}"

# Ground-truth stage id per level: BBCH id when present, else the Terres Inovia code.
def gt_stage_id(row, level):
    bbch = row.get(f"bbch_{level}_stage_id")
    if isinstance(bbch, str) and bbch.strip():
        return bbch.strip()
    ti = row.get("ti_stage_id")
    if level == "secondary" and isinstance(ti, str) and ti.strip():
        return ti.strip()
    return ""

for level in ("principal", "secondary"):
    df_bench[f"gt_{level}_id"] = df_bench.apply(lambda r: gt_stage_id(r, level), axis=1)

print("Photos with a principal gt id:", int((df_bench['gt_principal_id'] != '').sum()))
print("Photos with a secondary gt id:", int((df_bench['gt_secondary_id'] != '').sum()))
df_bench[["photo_name", "crop_name", "image_path","gt_principal_id", "gt_secondary_id"]].head(5)

Photos with a principal gt id: 93
Photos with a secondary gt id: 94


,photo_name,crop_name,image_path,gt_principal_id,gt_secondary_id
0,citrus#photo_citrus_bbch_00_1.jpg,citrus,/projects/agrifutur/sji/bsv-crossmodalIR/BBCH ...,0,00
1,citrus#photo_citrus_bbch_01_1.jpg,citrus,/projects/agrifutur/sji/bsv-crossmodalIR/BBCH ...,0,01
2,citrus#photo_citrus_bbch_03_1.jpg,citrus,/projects/agrifutur/sji/bsv-crossmodalIR/BBCH ...,0,03
3,citrus#photo_citrus_bbch_07_1.jpg,citrus,/projects/agrifutur/sji/bsv-crossmodalIR/BBCH ...,0,07
4,citrus#photo_citrus_bbch_07_2.jpeg,citrus,/projects/agrifutur/sji/bsv-crossmodalIR/BBCH ...,0,07


### Sanity check — every ground-truth id must exist in its crop's candidate pool

If a photo's stage id is not in the pool built by notebook 31, that photo can never be retrieved and would silently deflate the scores. This cell reports (and drops) any such photo.

In [6]:
pool_ids = {
    (crop, level): {r["stage_id"] for r in rows}
    for (crop, level), rows in stage_lists.items()
}

unreachable = []
for _, row in df_bench.iterrows():
    for level in ("principal", "secondary"):
        gt = row[f"bbch_{level}_stage_id"]
        if not gt:
            continue
        if gt not in pool_ids.get((row["crop_name"], level), set()):
            unreachable.append({"photo_name": row["photo_name"], "level": level, "gt_id": gt})

if unreachable:
    print(f"WARNING: {len(unreachable)} (photo, level) pairs have a gt id outside the candidate pool:")
    display(pd.DataFrame(unreachable))
else:
    print("OK: every ground-truth id is present in its crop's candidate pool.")

,photo_name,level,gt_id
0,oilseedFlax#terres_inovia_Zoe_LeBihan_F1_photo...,secondary,60
1,oilseedFlax#terres_inovia_Zoe_LeBihan_F5_photo...,secondary,62
2,oilseedRape#terres_inovia_E_photo_oilseed_rape...,principal,5
3,oilseedRape#terres_inovia_E_photo_oilseed_rape...,secondary,59


## Section 4 — Model loaders

Every model is wrapped in a small bundle exposing the same two methods, so the retrieval code below is architecture-agnostic:

- `encode_images(paths) -> (N, D)` — L2-normalized image embeddings
- `encode_texts(texts) -> list[(n_chunks, D)]` — L2-normalized text embeddings, one tensor per text

The list-of-tensors return type is what lets a multi-vector text encoder (SigLIP2's 64-token chunking) and a single-vector one (BGE-M3) share the same interface. Similarity is then `max` over chunks for multi-vector models and a plain dot product for single-vector ones — see `similarity_matrix` below.

Loaders:

| loader | encoders | heads | text handling | used by |
|---|---|---|---|---|
| `load_dinov3_bge` | DINOv3 + BGE-M3 (trained weights when present) | `head_image_dinov3` / `head_text_bge` (or `*_v2` for run 06) | single vector (CLS) | the notebook-27 / 28 / 29 sweeps |
| `load_siglip2_bge` | SigLIP2 for both towers (+ unfrozen `vision_model` / `text_model` weights when present) | `head_image_siglip2` / `head_text_siglip2` | `text_pooling="max"` (multi-vector) or `"mean"` (mean-pool + renorm) | the 27C / 27D sweeps |
| `load_siglip2_raw` | pretrained SigLIP2, **no projection heads** | — | `text_mode`: `chunks_max` (default) / `truncate` / `chunks_mean` | out-of-the-box reference, NB26 zero-shot variants |
| `load_dinov3_bge_raw` | pretrained DINOv3 + BGE-M3, **no trained heads** (fixed random projection) | — | single vector (CLS) | no-training floor for the DINOv3+BGE pair |
| `load_clip_raw` | pretrained CLIP (ViT-B/32), **no projection heads** | — | `text_mode`: `truncate` (default) / `chunks_mean` | E7 + NB26 zero-shot variants |
| `load_random` | none (random vectors) | — | — | metric sanity check (`zz_random`) |

`load_dinov3_bge` and `load_siglip2_bge` accept alternative head/encoder file names and an optional full `checkpoint_file` (`latest_checkpoint.pt`, which holds every tower plus the epoch and is preferred whenever present). Encoder file names may be given as a **list of candidates**, tried in order; a missing encoder file leaves the pretrained encoder in place (correct for heads-only runs) and the loader prints exactly which weights it loaded.

Text pooling **matches each model's training**: `27D_*` and the `26_zs_*_meanpool` baselines are scored with mean-pooled chunk embeddings (mean over the projected chunks, re-normalized — NB27D's `meanpool_text_chunks`), everything else keeps max-over-chunks or truncation as trained. `text_mode="truncate"` is NB26's literature-standard zero-shot: one vector from the first native-context tokens (64 for SigLIP2, 77 for CLIP).


In [7]:
class ProjectionHead(nn.Module):
    """Single linear layer into the shared space, L2-normalized. Identical to the training notebooks."""

    def __init__(self, in_dim, out_dim=SHARED_DIM):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return F.normalize(self.proj(x), p=2, dim=-1)


class ModelBundle:
    """Uniform interface over every evaluated model.

    encode_images(paths) -> (N, 512) tensor
    encode_texts(texts)  -> list of (n_chunks, 512) tensors, one per text
    """

    def __init__(self, name, encode_images, encode_texts, multi_vector_text=False):
        self.name = name
        self.encode_images = encode_images
        self.encode_texts = encode_texts
        self.multi_vector_text = multi_vector_text

    def __repr__(self):
        return f"ModelBundle({self.name})"


def _load_state(path, module, device):
    """Load a state dict, tolerating the 'module.' prefix from DataParallel checkpoints."""
    state = torch.load(path, map_location=device)
    if isinstance(state, dict) and all(k.startswith("module.") for k in state):
        state = {k[len("module."):]: v for k, v in state.items()}
    module.load_state_dict(state)
    return module


print("ProjectionHead + ModelBundle ready.")

ProjectionHead + ModelBundle ready.


In [8]:
def _first_existing(run_dir, candidates):
    """Return the first existing file among `candidates` (str or list of str), else None."""
    if candidates is None:
        return None
    if isinstance(candidates, str):
        candidates = [candidates]
    for name in candidates:
        path = run_dir / name
        if path.exists():
            return path
    return None


def _strip_module_prefix(state):
    if isinstance(state, dict) and state and all(k.startswith("module.") for k in state):
        return {k[len("module."):]: v for k, v in state.items()}
    return state


def load_dinov3_bge(run_dir, name, head_image_file="head_image_dinov3_v2.pt",
                    head_text_file="head_text_bge_v2.pt",
                    encoder_image_file="dinov3_model_unfrozen.pt",
                    encoder_text_file="bge_model_unfrozen.pt",
                    checkpoint_file=None,
                    checkpoint_keys=None):
    """DINOv3 (image) + BGE-M3 (text) with trained projection heads.

    Used for the single-run folders (06/07) and, with the notebook-29 / Smaller_train_sizes
    file names, for the size sweeps.

    Two ways to supply the trained weights, in priority order:

    1. `checkpoint_file` (e.g. "latest_checkpoint.pt") — the resume notebook's full state
       dict, which holds EVERY tower plus the epoch it was saved at. This is authoritative
       and is preferred whenever it exists, because the per-size folders are inconsistent
       about which individual files they kept (`*_unfrozen.pt` for some sizes, `*_model.pt`
       for others, neither for the smallest). The epoch is printed, so the checkpoint a
       score came from is never ambiguous.
    2. Individual files — heads from `head_image_file` / `head_text_file`, encoders from
       `encoder_image_file` / `encoder_text_file` (a single name or a LIST of candidates
       tried in order). Missing encoder files leave the pretrained encoder in place, which
       is correct for the smallest sizes (heads-only training).

    Whatever was loaded is printed, so a silent fallback to pretrained encoders is visible.
    """
    run_dir = Path(run_dir)

    dinov3_processor = AutoImageProcessor.from_pretrained(DINOV3_MODEL_NAME, cache_dir=HF_CACHE)
    dinov3_model = AutoModel.from_pretrained(DINOV3_MODEL_NAME,cache_dir=HF_CACHE).to(device).eval()
    dinov3_dim = dinov3_model.config.hidden_size

    bge_tokenizer = AutoTokenizer.from_pretrained(BGE_MODEL_NAME,cache_dir=HF_CACHE)
    bge_model = AutoModel.from_pretrained(BGE_MODEL_NAME,cache_dir=HF_CACHE).to(device).eval()
    bge_dim = bge_model.config.hidden_size

    head_image = ProjectionHead(dinov3_dim).to(device).eval()
    head_text = ProjectionHead(bge_dim).to(device).eval()

    ckpt_path = _first_existing(run_dir, checkpoint_file)
    if ckpt_path is not None:
        # Authoritative path: one file, every tower, plus the epoch.
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        keys = checkpoint_keys or {
            "image_encoder": "dinov3_model",
            "text_encoder": "bge_model",
            "head_image": "head_image_dinov3",
            "head_text": "head_text_bge",
        }
        targets = {
            "image_encoder": dinov3_model,
            "text_encoder": bge_model,
            "head_image": head_image,
            "head_text": head_text,
        }
        print(f"[{name}] loading ALL weights from {ckpt_path.name} "
              f"(epoch {ckpt.get('epoch', '?')}, step {ckpt.get('global_step', '?')})")
        for role, module in targets.items():
            key = keys[role]
            if key not in ckpt:
                raise KeyError(f"[{name}] {ckpt_path.name} has no '{key}' entry; "
                               f"available: {list(ckpt.keys())}")
            module.load_state_dict(_strip_module_prefix(ckpt[key]))
            print(f"[{name}]   {role:13s} <- {ckpt_path.name}['{key}']")
    else:
        # Fallback: individual files. Encoders may legitimately be absent (heads-only runs).
        enc_img_path = _first_existing(run_dir, encoder_image_file)
        if enc_img_path is not None:
            _load_state(enc_img_path, dinov3_model, device)
            print(f"[{name}] image encoder <- {enc_img_path.name}")
        else:
            print(f"[{name}] image encoder <- PRETRAINED (no {encoder_image_file} in {run_dir.name})")

        enc_txt_path = _first_existing(run_dir, encoder_text_file)
        if enc_txt_path is not None:
            _load_state(enc_txt_path, bge_model, device)
            print(f"[{name}] text encoder  <- {enc_txt_path.name}")
        else:
            print(f"[{name}] text encoder  <- PRETRAINED (no {encoder_text_file} in {run_dir.name})")

        _load_state(run_dir / head_image_file, head_image, device)
        _load_state(run_dir / head_text_file, head_text, device)
        print(f"[{name}] heads        <- {head_image_file}, {head_text_file}")

    @torch.no_grad()
    def encode_images(paths):
        chunks = []
        for i in range(0, len(paths), IMAGE_ENCODE_CHUNK_SIZE):
            batch = [Image.open(p).convert("RGB") for p in paths[i:i + IMAGE_ENCODE_CHUNK_SIZE]]
            inputs = dinov3_processor(images=batch, return_tensors="pt").to(device)
            out = dinov3_model(**inputs)
            chunks.append(out.last_hidden_state[:, 0, :])   # CLS token
        raw = torch.cat(chunks, dim=0)
        return head_image(raw)

    @torch.no_grad()
    def encode_texts(texts):
        inputs = bge_tokenizer(texts, return_tensors="pt", padding=True, truncation=True,
                               max_length=BGE_MAX_LENGTH).to(device)
        out = bge_model(**inputs)
        raw = out.last_hidden_state[:, 0, :]                # CLS pooling
        emb = head_text(raw)
        return [emb[i:i + 1] for i in range(emb.shape[0])]  # one (1, 512) tensor per text

    return ModelBundle(name, encode_images, encode_texts, multi_vector_text=False)


print("load_dinov3_bge() ready.")

load_dinov3_bge() ready.


In [9]:
def load_siglip2_bge(run_dir, name,
                     head_image_file="head_image_siglip2.pt",
                     head_text_file="head_text_siglip2.pt",
                     encoder_image_file="siglip2_vision_unfrozen.pt",
                     encoder_text_file="siglip2_text_unfrozen.pt",
                     checkpoint_file=None,
                     checkpoint_keys=None,
                     text_pooling="max"):
    """SigLIP2 (image + chunked text) with trained projection heads — the NB27C/27D sweeps.

    The sweeps keep `head_*_siglip2.pt`, optionally unfrozen `vision_model` / `text_model`
    towers, and `latest_checkpoint.pt`; all file names are overridable.

    Weights, in priority order:

    1. `checkpoint_file` (e.g. "latest_checkpoint.pt") — the training notebook's full state
       dict, holding every tower plus the epoch. Authoritative whenever present.
    2. Individual files — heads from `head_image_file` / `head_text_file`, encoders from
       `encoder_image_file` / `encoder_text_file` (a single name or a LIST of candidates
       tried in order). Missing encoder files leave the pretrained towers in place (correct
       for frozen-encoder runs like 27C/27D's last_0_layers).

    SigLIP2's text tower accepts only 64 tokens, so each text is split into 64-token chunks.
    `text_pooling` decides how the chunks become one representation, matching training:

    - "max"  (default) — every chunk is projected and kept; similarity takes the max over
      chunks (NB27C).
    - "mean" — the projected chunks are mean-pooled and re-normalized into a single vector
      (NB27D's `meanpool_text_chunks`), so the bundle is single-vector.

    Whatever was loaded is printed, so a silent fallback to pretrained encoders is visible.
    """
    if text_pooling not in ("max", "mean"):
        raise ValueError(f"text_pooling must be 'max' or 'mean', got {text_pooling!r}")
    run_dir = Path(run_dir)

    siglip_processor = AutoProcessor.from_pretrained(SIGLIP2_MODEL_NAME,cache_dir=HF_CACHE)
    siglip_model = AutoModel.from_pretrained(SIGLIP2_MODEL_NAME,cache_dir=HF_CACHE).to(device).eval()
    vision_model = siglip_model.vision_model
    text_model = siglip_model.text_model
    siglip_tokenizer = siglip_processor.tokenizer
    siglip_dim = siglip_model.config.vision_config.hidden_size

    head_image = ProjectionHead(siglip_dim).to(device).eval()
    head_text = ProjectionHead(siglip_dim).to(device).eval()

    ckpt_path = _first_existing(run_dir, checkpoint_file)
    if ckpt_path is not None:
        # Authoritative path: one file, every tower, plus the epoch.
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        keys = checkpoint_keys or {
            "image_encoder": "vision_model",
            "text_encoder": "text_model",
            "head_image": "head_image_siglip2",
            "head_text": "head_text_siglip2",
        }
        targets = {
            "image_encoder": vision_model,
            "text_encoder": text_model,
            "head_image": head_image,
            "head_text": head_text,
        }
        print(f"[{name}] loading ALL weights from {ckpt_path.name} "
              f"(epoch {ckpt.get('epoch', '?')}, step {ckpt.get('global_step', '?')})")
        for role, module in targets.items():
            key = keys[role]
            if key not in ckpt:
                raise KeyError(f"[{name}] {ckpt_path.name} has no '{key}' entry; "
                               f"available: {list(ckpt.keys())}")
            module.load_state_dict(_strip_module_prefix(ckpt[key]))
            print(f"[{name}]   {role:13s} <- {ckpt_path.name}['{key}']")
    else:
        # Fallback: individual files. Encoders may legitimately be absent (frozen runs).
        enc_img_path = _first_existing(run_dir, encoder_image_file)
        if enc_img_path is not None:
            _load_state(enc_img_path, vision_model, device)
            print(f"[{name}] image encoder <- {enc_img_path.name}")
        else:
            print(f"[{name}] image encoder <- PRETRAINED (no {encoder_image_file} in {run_dir.name})")

        enc_txt_path = _first_existing(run_dir, encoder_text_file)
        if enc_txt_path is not None:
            _load_state(enc_txt_path, text_model, device)
            print(f"[{name}] text encoder  <- {enc_txt_path.name}")
        else:
            print(f"[{name}] text encoder  <- PRETRAINED (no {encoder_text_file} in {run_dir.name})")

        _load_state(run_dir / head_image_file, head_image, device)
        _load_state(run_dir / head_text_file, head_text, device)
        print(f"[{name}] heads        <- {head_image_file}, {head_text_file}")

    def _pool(out):
        return out.pooler_output if hasattr(out, "pooler_output") else out

    def chunk_text(text, chunk_tokens=SIGLIP2_TEXT_CHUNK_TOKENS):
        ids = siglip_tokenizer.encode(text, add_special_tokens=False)
        if not ids:
            return [text]
        return [siglip_tokenizer.decode(ids[i:i + chunk_tokens])
                for i in range(0, len(ids), chunk_tokens)]

    @torch.no_grad()
    def encode_images(paths):
        chunks = []
        for i in range(0, len(paths), IMAGE_ENCODE_CHUNK_SIZE):
            batch = [Image.open(p).convert("RGB") for p in paths[i:i + IMAGE_ENCODE_CHUNK_SIZE]]
            inputs = siglip_processor(images=batch, return_tensors="pt").to(device)
            chunks.append(_pool(siglip_model.get_image_features(**inputs)))
        raw = torch.cat(chunks, dim=0)
        return head_image(raw)

    @torch.no_grad()
    def encode_texts(texts):
        out = []
        for text in texts:
            chunks = chunk_text(text)
            inputs = siglip_processor(text=chunks, padding="max_length",
                                      max_length=SIGLIP2_TEXT_CHUNK_TOKENS,
                                      truncation=True, return_tensors="pt").to(device)
            raw = _pool(siglip_model.get_text_features(**inputs))
            projected = head_text(raw)                    # (n_chunks, 512), each L2-normalized
            if text_pooling == "mean":
                # NB27D's meanpool_text_chunks: mean over projected chunks, re-normalize.
                out.append(F.normalize(projected.mean(dim=0), p=2, dim=0).unsqueeze(0))
            else:
                out.append(projected)                     # (n_chunks, 512)
        return out

    multi_vector = text_pooling == "max"
    print(f"[{name}] text_pooling={text_pooling} "
          f"({'multi-vector, max over chunks' if multi_vector else 'single vector, mean-pooled'})")
    return ModelBundle(name, encode_images, encode_texts, multi_vector_text=multi_vector)


print("load_siglip2_bge() ready.")


load_siglip2_bge() ready.


In [10]:
def load_siglip2_raw(name, text_mode="chunks_max"):
    """Pretrained SigLIP2 with NO projection heads — the out-of-the-box reference.

    Both towers emit 768-dim embeddings in SigLIP2's own space, so cosine similarity between
    image and text is used directly. No checkpoint involved.

    `text_mode` selects how long texts become vectors (NB26's zero-shot variants):

    - "chunks_max"  (default) — split into 64-token chunks, keep the chunks as a
      multi-vector; similarity takes the max over chunks.
    - "truncate"    — one vector from the first 64 tokens (NB26's literature-standard
      zero-shot: SigLIP2 was trained on short captions).
    - "chunks_mean" — chunk at 64 tokens, encode, mean-pool + re-normalize into one vector
      (NB26's mean-pool baseline).
    """
    if text_mode not in ("chunks_max", "truncate", "chunks_mean"):
        raise ValueError(f"text_mode must be chunks_max|truncate|chunks_mean, got {text_mode!r}")
    siglip_processor = AutoProcessor.from_pretrained(SIGLIP2_MODEL_NAME, cache_dir=HF_CACHE)
    siglip_model = AutoModel.from_pretrained(SIGLIP2_MODEL_NAME, cache_dir=HF_CACHE).to(device).eval()
    siglip_tokenizer = siglip_processor.tokenizer

    def _pool(out):
        return out.pooler_output if hasattr(out, "pooler_output") else out

    def chunk_text(text, chunk_tokens=SIGLIP2_TEXT_CHUNK_TOKENS):
        ids = siglip_tokenizer.encode(text, add_special_tokens=False)
        if not ids:
            return [text]
        return [siglip_tokenizer.decode(ids[i:i + chunk_tokens])
                for i in range(0, len(ids), chunk_tokens)]

    @torch.no_grad()
    def encode_images(paths):
        chunks = []
        for i in range(0, len(paths), IMAGE_ENCODE_CHUNK_SIZE):
            batch = [Image.open(p).convert("RGB") for p in paths[i:i + IMAGE_ENCODE_CHUNK_SIZE]]
            inputs = siglip_processor(images=batch, return_tensors="pt").to(device)
            chunks.append(F.normalize(_pool(siglip_model.get_image_features(**inputs)), p=2, dim=-1))
        return torch.cat(chunks, dim=0)

    @torch.no_grad()
    def encode_texts(texts):
        out = []
        for text in texts:
            if text_mode == "truncate":
                pieces = [text]                           # one vector, first 64 tokens
            else:
                pieces = chunk_text(text)
            inputs = siglip_processor(text=pieces, padding="max_length",
                                      max_length=SIGLIP2_TEXT_CHUNK_TOKENS,
                                      truncation=True, return_tensors="pt").to(device)
            raw = F.normalize(_pool(siglip_model.get_text_features(**inputs)), p=2, dim=-1)
            if text_mode == "chunks_mean":
                pooled = F.normalize(raw.mean(dim=0), p=2, dim=0)   # NB26 mean-pool baseline
                out.append(pooled.unsqueeze(0))
            else:
                out.append(raw)
        return out

    multi_vector = text_mode == "chunks_max"
    print(f"[{name}] text_mode={text_mode}")
    return ModelBundle(name, encode_images, encode_texts, multi_vector_text=multi_vector)


print("load_siglip2_raw() ready.")


load_siglip2_raw() ready.


In [24]:
def similarity_matrix(image_emb, text_embs, multi_vector_text=False):
    """(N_images, N_texts) cosine similarity.

    image_emb: (N, D) normalized. text_embs: list of (n_chunks, D) normalized tensors.
    Single-vector models give a plain dot product; multi-vector models take the max over
    the text's chunks (the text matches if any chunk matches well).
    """
    image_emb = image_emb.cpu()
    text_embs = [t.cpu() for t in text_embs]
    
    if not multi_vector_text:
        text_mat = torch.cat(text_embs, dim=0)              # (N_texts, D)
        return image_emb @ text_mat.T

    sim = torch.empty((image_emb.shape[0], len(text_embs)))
    for j, chunks in enumerate(text_embs):
        sim[:, j] = (image_emb @ chunks.T).max(dim=1).values
    return sim


def compute_retrieval_metrics(sim_matrix, k_values=K_VALUES):
    """R@K and MRR for image -> text retrieval.

    Row i's correct answer is column i, so the rank of the true match is the number of
    candidates scoring at least as high as it. Identical to the protocol used in Finetuning/.
    """
    n = sim_matrix.shape[0]
    true_scores = sim_matrix.diag().unsqueeze(1)
    ranks = (sim_matrix >= true_scores).sum(dim=1)
    results = {}
    for k in k_values:
        results[f"R@{k}"] = (ranks <= k).float().mean().item()
    results["MRR"] = (1.0 / ranks.float()).mean().item()
    results["N"] = n
    return results


def random_baseline_metrics(n_candidates, k_values=K_VALUES):
    """Expected R@K / MRR for a random ranking over n_candidates (chance level)."""
    results = {}
    for k in k_values:
        results[f"R@{k}"] = min(k, n_candidates) / n_candidates
    results["MRR"] = sum(1.0 / r for r in range(1, n_candidates + 1)) / n_candidates
    results["N"] = n_candidates
    return results


print("similarity_matrix() + compute_retrieval_metrics() + random_baseline_metrics() ready.")

similarity_matrix() + compute_retrieval_metrics() + random_baseline_metrics() ready.


In [11]:
def load_random(name, dim=SHARED_DIM, seed=0):
    """Random-embedding control: no model, no weights, no checkpoint.

    Empirical counterpart of random_baseline_metrics(). It exercises the whole pipeline
    (similarity_matrix, chunk handling, rank_metrics_with_targets, gt -> column mapping),
    so if its scores do not land near the analytic chance level, the metric code is wrong.
    """
    g = torch.Generator().manual_seed(seed)

    def encode_images(paths):
        return F.normalize(torch.randn(len(paths), dim, generator=g), p=2, dim=-1)

    def encode_texts(texts):
        return [F.normalize(torch.randn(1, dim, generator=g), p=2, dim=-1) for _ in texts]

    return ModelBundle(name, encode_images, encode_texts, multi_vector_text=False)


print("load_random() ready.")

load_random() ready.


In [12]:
def load_clip_raw(name, text_mode="truncate"):
    """Pretrained CLIP (ViT-B/32) with NO projection heads — E7, second out-of-box family.

    Unlike DINOv3+BGE, CLIP's image and text towers were contrastively pretrained TOGETHER,
    so their embeddings already share one space: a plain dot product is meaningful and no
    projection or training is needed. This is the fair out-of-box comparison for SigLIP2 —
    it separates "SigLIP2 is special" from "any natively-aligned pair works out of the box".

    CLIP's text tower accepts at most 77 tokens. `text_mode` selects how longer texts are
    handled (NB26's zero-shot variants):

    - "truncate"    (default) — one vector from the first 77 tokens. CLIP was trained on
      short captions, so this is the literature-standard zero-shot. Truncation is counted
      and reported, never silent.
    - "chunks_mean" — chunk at 77 tokens, encode, mean-pool + re-normalize into one vector
      (NB26's mean-pool baseline).

    NOTE: this checkpoint lives in CLIP_CACHE (hf_cache), not HF_CACHE (hf_cache2).
    """
    if text_mode not in ("truncate", "chunks_mean"):
        raise ValueError(f"text_mode must be truncate|chunks_mean, got {text_mode!r}")
    clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME, cache_dir=CLIP_CACHE)
    clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME, cache_dir=CLIP_CACHE).to(device).eval()
    clip_tokenizer = clip_processor.tokenizer

    n_truncated = {"count": 0}

    def chunk_text(text, chunk_tokens=CLIP_MAX_LENGTH):
        ids = clip_tokenizer.encode(text, add_special_tokens=False)
        if not ids:
            return [text]
        return [clip_tokenizer.decode(ids[i:i + chunk_tokens])
                for i in range(0, len(ids), chunk_tokens)]

    @torch.no_grad()
    def encode_images(paths):
        chunks = []
        for i in range(0, len(paths), IMAGE_ENCODE_CHUNK_SIZE):
            batch = [Image.open(p).convert("RGB") for p in paths[i:i + IMAGE_ENCODE_CHUNK_SIZE]]
            inputs = clip_processor(images=batch, return_tensors="pt").to(device)
            emb = clip_model.get_image_features(**inputs)
            chunks.append(F.normalize(emb, p=2, dim=-1))
        return torch.cat(chunks, dim=0)

    @torch.no_grad()
    def encode_texts(texts):
        out = []
        for text in texts:
            if text_mode == "truncate":
                pieces = [text]                           # one vector, first 77 tokens
                # Count how many texts exceed CLIP's 77-token window, so truncation is not silent.
                if len(clip_tokenizer.encode(text, add_special_tokens=True)) > CLIP_MAX_LENGTH:
                    n_truncated["count"] += 1
            else:
                pieces = chunk_text(text)
            inputs = clip_processor(text=pieces, padding="max_length",
                                    max_length=CLIP_MAX_LENGTH,
                                    truncation=True, return_tensors="pt").to(device)
            emb = F.normalize(clip_model.get_text_features(**inputs), p=2, dim=-1)
            if text_mode == "chunks_mean":
                pooled = F.normalize(emb.mean(dim=0), p=2, dim=0)    # NB26 mean-pool baseline
                out.append(pooled.unsqueeze(0))
            else:
                out.append(emb)
        return out

    print(f"[{name}] CLIP loaded from {CLIP_CACHE}; text_mode={text_mode}"
          + (f"; {n_truncated['count']} text(s) truncated at {CLIP_MAX_LENGTH} tokens"
             if text_mode == "truncate" else ""))
    return ModelBundle(name, encode_images, encode_texts,
                       multi_vector_text=False)


print("load_clip_raw() ready.")


load_clip_raw() ready.


## Section 5 — Model registry

Builds the list of models to evaluate. Each entry is `(model_id, loader_callable)`; the loader is only invoked when the model's embeddings are not already cached.

Sweep entries (one per family: the notebook-27/27C/27D layer sweeps, the notebook-28 `smaller_size_*` size sweep, the notebook-29 mining sweep) are added only for values whose run folder actually holds weights, so a partial sweep still runs and untrained values (e.g. `last_8_layers`) are reported as skipped rather than failing.


In [18]:
MODEL_REGISTRY = []   # list of (model_id, loader)
SKIPPED = []          # list of (model_id, reason)


def register(model_id, loader):
    MODEL_REGISTRY.append((model_id, loader))


def skip(model_id, reason):
    SKIPPED.append((model_id, reason))


def make_loader(loader_kind, run_dir, model_id, files=None, text_mode=None, text_pooling=None):
    """Build the loader callable for one (loader family, checkpoint dir) pair."""
    files = files or {}
    if loader_kind == "siglip2_raw":
        return lambda: load_siglip2_raw(model_id, text_mode=text_mode or "chunks_max")
    if loader_kind == "random":
        return lambda: load_random(model_id)
    if loader_kind == "dinov3_bge":
        return lambda: load_dinov3_bge(
            run_dir, model_id,
            head_image_file=files.get("head_image", "head_image_dinov3_v2.pt"),
            head_text_file=files.get("head_text", "head_text_bge_v2.pt"),
            encoder_image_file=files.get("encoder_image", "dinov3_model_unfrozen.pt"),
            encoder_text_file=files.get("encoder_text", "bge_model_unfrozen.pt"),
            checkpoint_file=files.get("checkpoint"),
        )
    if loader_kind == "siglip2_bge":
        return lambda: load_siglip2_bge(
            run_dir, model_id,
            head_image_file=files.get("head_image", "head_image_siglip2.pt"),
            head_text_file=files.get("head_text", "head_text_siglip2.pt"),
            encoder_image_file=files.get("encoder_image", "siglip2_vision_unfrozen.pt"),
            encoder_text_file=files.get("encoder_text", "siglip2_text_unfrozen.pt"),
            checkpoint_file=files.get("checkpoint"),
            text_pooling=text_pooling or "max",
        )
    if loader_kind == "dinov3_bge_raw":
        return lambda: load_dinov3_bge_raw(model_id)
    if loader_kind == "clip_raw":
        return lambda: load_clip_raw(model_id, text_mode=text_mode or "truncate")
    raise ValueError(f"unknown loader family: {loader_kind!r}")


# --- configured single-run models ------------------------------------------
for spec in MODELS:
    model_id = spec["id"]
    loader_kind = spec["loader"]
    run_dir = spec["dir"]

    if loader_kind in ("siglip2_raw", "random", "dinov3_bge_raw", "clip_raw"):
        register(model_id, make_loader(loader_kind, None, model_id, spec.get("files"),
                                       text_mode=spec.get("text_mode")))
        continue

    if run_dir is None:
        skip(model_id, "no checkpoint dir and loader needs one")
        continue
    if not Path(run_dir).exists():
        skip(model_id, f"checkpoint folder missing: {run_dir}")
        continue

    try:
        loader = make_loader(loader_kind, Path(run_dir), model_id, spec.get("files"),
                             text_mode=spec.get("text_mode"),
                             text_pooling=spec.get("text_pooling"))
    except ValueError as exc:
        skip(model_id, str(exc))
        continue
    register(model_id, loader)


def register_sweep(sweep):
    """Register one sweep family. A value is only registered when its weights exist.

    A value qualifies when its run folder has EITHER the full `checkpoint` file OR both
    head files. Encoder files are optional (frozen-encoder runs save heads only), so they
    are not required here; the loaders report which weights they actually loaded.
    """
    if not sweep:
        return
    files = sweep.get("files", {})
    head_img_name = files.get("head_image", "head_image_dinov3.pt")
    head_txt_name = files.get("head_text", "head_text_bge.pt")
    ckpt_name = files.get("checkpoint")
    dir_template = sweep.get("dir_template", "size_{value}")
    for value in sweep["values"]:
        model_id = f"{sweep['id_prefix']}_{value}"
        run_dir = Path(sweep["root"]) / dir_template.format(value=value)
        has_ckpt = bool(ckpt_name) and (run_dir / ckpt_name).exists()
        has_heads = (run_dir / head_img_name).exists() and (run_dir / head_txt_name).exists()
        if not (has_ckpt or has_heads):
            skip(model_id, f"no weights in {run_dir}")
            continue
        register(model_id, make_loader(sweep["loader"], run_dir, model_id, files,
                                       text_pooling=sweep.get("text_pooling")))


# --- sweep families (notebook-27/27C/27D layers, 28 sizes, 29 mining sizes) --
for sweep in SWEEPS:
    register_sweep(sweep)

# --- report ----------------------------------------------------------------
print(f"Registered {len(MODEL_REGISTRY)} model(s):")
for model_id, _ in MODEL_REGISTRY:
    print("  +", model_id)

if SKIPPED:
    print(f"\nSkipped {len(SKIPPED)} model(s):")
    for model_id, reason in SKIPPED:
        print(f"  - {model_id}: {reason}")
else:
    print("\nNothing skipped.")

assert MODEL_REGISTRY, "No model registered -- populate MODELS (or SWEEPS) in the config cell."


Registered 37 model(s):
  + 00_siglip2_out_of_box
  + 26_zs_siglip2_trunc
  + 26_zs_siglip2_meanpool
  + 26_zs_clip_meanpool
  + zz_random
  + zz_dinov3_bge_raw
  + zz_clip_raw
  + 29_sweep_size_100
  + 29_sweep_size_500
  + 29_sweep_size_1000
  + 29_sweep_size_2500
  + 29_sweep_size_5000
  + 29_sweep_size_10000
  + 29_sweep_size_20000
  + 29_sweep_size_33208
  + smaller_size_100
  + smaller_size_500
  + smaller_size_1000
  + smaller_size_2500
  + smaller_size_5000
  + smaller_size_10000
  + smaller_size_20000
  + smaller_size_33526
  + 27_last_0
  + 27_last_2
  + 27_last_4
  + 27_last_6
  + 27_last_8
  + 27C_last_0
  + 27C_last_2
  + 27C_last_4
  + 27C_last_6
  + 27C_last_8
  + 27D_last_0
  + 27D_last_2
  + 27D_last_4
  + 27D_last_8

Skipped 1 model(s):
  - 27D_last_6: no weights in /projects/agrifutur/sji/cross_encoder_ft/diff_unfrozen_layers/textimage_siglip2_meanpool/last_6_layers


## Section 6 — Encode the benchmark (cached)

For each model, encodes:

- every benchmark photo (once — the same image embeddings serve all three tasks)
- the crop-name texts (task 1 candidates)
- every stage description, per crop / level / language (task 2 candidates)

Results are cached to `vars/bbch_emb_<model_id>.pkl`. Re-running the notebook reuses the cache, so only newly registered models are encoded.

In [19]:
def build_candidate_texts():
    """All texts the benchmark ranks against, grouped by task.

    Returns
    -------
    crop_texts : list[str]                     task 1 candidates (crop names)
    stage_texts : dict[(crop, level, lang)] -> list[str]   task 2 candidates
    """
    crop_texts = list(crop_candidates)

    stage_texts = {}
    for (crop, level), rows in stage_lists.items():
        for lang in ("fr", "en"):
            stage_texts[(crop, level, lang)] = [r[f"desc_{lang}"] for r in rows]
    return crop_texts, stage_texts


crop_texts, stage_texts = build_candidate_texts()
print(f"Crop candidates: {len(crop_texts)}")
print(f"Stage candidate groups: {len(stage_texts)}")
for key in sorted(stage_texts):
    print(f"  {key}: {len(stage_texts[key])} candidates")

Crop candidates: 32
Stage candidate groups: 10
  ('citrus', 'principal', 'en'): 8 candidates
  ('citrus', 'principal', 'fr'): 8 candidates
  ('citrus', 'secondary', 'en'): 35 candidates
  ('citrus', 'secondary', 'fr'): 35 candidates
  ('oilseedFlax', 'principal', 'en'): 9 candidates
  ('oilseedFlax', 'principal', 'fr'): 9 candidates
  ('oilseedFlax', 'secondary', 'en'): 81 candidates
  ('oilseedFlax', 'secondary', 'fr'): 81 candidates
  ('oilseedRape', 'secondary', 'en'): 28 candidates
  ('oilseedRape', 'secondary', 'fr'): 28 candidates


In [20]:
def encode_model(model_id, loader, force=False):
    """Encode the benchmark with one model, caching the result to vars/.

    `force` (or the model's `recompute` config flag) bypasses and overwrites the cache —
    use it when an id's meaning changed (e.g. smaller_size_* now points at the NB28 runs),
    otherwise the stale cache would silently win.

    Returns a dict with the image embeddings and every candidate-text embedding group.
    """
    cache_file = f"bbch_emb_{model_id}.pkl"
    if cache_exists(cache_file) and not force and model_id not in RECOMPUTE:
        print(f"[{model_id}] loading cached embeddings")
        return reload_vars(cache_file)
    if cache_exists(cache_file) and (force or model_id in RECOMPUTE):
        print(f"[{model_id}] re-encoding (recompute) -- overwriting vars/{cache_file}")

    print(f"[{model_id}] loading model ...")
    bundle = loader()

    print(f"[{model_id}] encoding {len(df_bench)} images ...")
    image_emb = bundle.encode_images(df_bench["image_path"].tolist()).cpu()

    print(f"[{model_id}] encoding {len(crop_texts)} crop texts ...")
    crop_emb = [t.cpu() for t in bundle.encode_texts(crop_texts)]

    stage_emb = {}
    for key, texts in stage_texts.items():
        stage_emb[key] = [t.cpu() for t in bundle.encode_texts(texts)]

    payload = {
        "model_id": model_id,
        "multi_vector_text": bundle.multi_vector_text,
        "image_emb": image_emb,
        "crop_emb": crop_emb,
        "stage_emb": stage_emb,
    }
    pkl_vars(payload, cache_file)
    print(f"[{model_id}] cached to vars/{cache_file}")

    del bundle
    if device == "cuda":
        torch.cuda.empty_cache()
    return payload


print("encode_model() ready.")


encode_model() ready.


In [21]:
# Encode every registered model (cached after the first run; RECOMPUTE ids always re-encode).
if RECOMPUTE:
    print(f"Recompute (cache ignored): {sorted(RECOMPUTE)}")

embeddings = {}
for model_id, loader in MODEL_REGISTRY:
    embeddings[model_id] = encode_model(model_id, loader)

print(f"\nEncoded {len(embeddings)} models.")
for model_id, payload in embeddings.items():
    print(f"  {model_id}: image_emb {tuple(payload['image_emb'].shape)}, "
          f"multi_vector_text={payload['multi_vector_text']}")


[00_siglip2_out_of_box] loading cached embeddings
[26_zs_siglip2_trunc] loading cached embeddings
[26_zs_siglip2_meanpool] loading cached embeddings
[26_zs_clip_meanpool] loading cached embeddings
[zz_random] loading cached embeddings
[zz_dinov3_bge_raw] loading cached embeddings
[zz_clip_raw] loading cached embeddings
[29_sweep_size_100] loading cached embeddings
[29_sweep_size_500] loading cached embeddings
[29_sweep_size_1000] loading cached embeddings
[29_sweep_size_2500] loading cached embeddings
[29_sweep_size_5000] loading cached embeddings
[29_sweep_size_10000] loading cached embeddings
[29_sweep_size_20000] loading cached embeddings
[29_sweep_size_33208] loading cached embeddings
[smaller_size_100] loading cached embeddings
[smaller_size_500] loading cached embeddings
[smaller_size_1000] loading cached embeddings
[smaller_size_2500] loading cached embeddings
[smaller_size_5000] loading cached embeddings
[smaller_size_10000] loading cached embeddings
[smaller_size_20000] loadin

## Section 7 — Task 1: image → crop

Every photo is ranked against the crop-name candidate list. Unlike the stage tasks, the correct column is not the diagonal (several photos share a crop), so a dedicated rank computation looks up each row's own ground-truth column.

Reported: overall R@K / MRR with the random-chance level for reference, plus a per-crop breakdown.

In [22]:
def rank_metrics_with_targets(sim_rows, target_idx, k_values=K_VALUES):
    """R@K / MRR when each row's correct candidate column is given explicitly.

    sim_rows:   (N, N_candidates) similarity tensor
    target_idx: (N,) long tensor of correct column indices (may repeat across rows)
    """
    true_scores = sim_rows.gather(1, target_idx.unsqueeze(1))      # (N, 1)
    ranks = (sim_rows >= true_scores).sum(dim=1)                   # 1-based rank of the truth
    results = {f"R@{k}": (ranks <= k).float().mean().item() for k in k_values}
    results["MRR"] = (1.0 / ranks.float()).mean().item()
    results["N"] = sim_rows.shape[0]
    return results


def prediction_rows_with_targets(sim_rows, target_idx, candidate_ids, row_metadata,
                                 top_k=PREDICTION_TOP_K):
    """Build one prediction row per photo, using the same tie-aware rank as the metrics."""
    rows = []
    for i, metadata in enumerate(row_metadata):
        scores = sim_rows[i]
        target = int(target_idx[i])
        true_score = scores[target]
        gt_rank = int((scores >= true_score).sum().item())
        order = torch.argsort(scores, descending=True)[:top_k].tolist()
        rows.append({
            **metadata,
            "gt_id": str(candidate_ids[target]),
            "predicted_id": str(candidate_ids[int(scores.argmax().item())]),
            "gt_rank": gt_rank,
            "is_correct": gt_rank == 1,
            "gt_similarity": float(true_score.item()),
            "predicted_similarity": float(scores.max().item()),
            "top_candidate_ids": json.dumps(
                [str(candidate_ids[j]) for j in order], ensure_ascii=False
            ),
            "top_candidate_scores": json.dumps(
                [float(scores[j].item()) for j in order]
            ),
        })
    return rows


print("rank_metrics_with_targets() + prediction_rows_with_targets() ready.")

rank_metrics_with_targets() + prediction_rows_with_targets() ready.


In [25]:
def evaluate_image_to_crop(model_id, payload):
    """Rank every photo against crop names; return metrics and per-photo predictions."""
    image_emb = payload["image_emb"]
    sim = similarity_matrix(image_emb, payload["crop_emb"], payload["multi_vector_text"])

    crop_to_idx = {c: i for i, c in enumerate(crop_texts)}
    target_idx = torch.tensor([crop_to_idx[c] for c in df_bench["crop_name"]], dtype=torch.long)

    overall = rank_metrics_with_targets(sim, target_idx)
    overall = {"model_id": model_id, "task": "image->crop", **overall}

    # per-crop breakdown: rank each crop's photos within the same global ranking
    rows = []
    for crop, idx in df_bench.groupby("crop_name").groups.items():
        idx = list(idx)
        sub_sim = sim[idx]
        sub_target = target_idx[idx]
        m = rank_metrics_with_targets(sub_sim, sub_target)
        rows.append({"model_id": model_id, "crop_name": crop, **m})

    row_metadata = [
        {
            "model_id": model_id,
            "task": "image->crop",
            "level": "crop",
            "lang": "",
            "photo_name": row["photo_name"],
            "crop_name": row["crop_name"],
            "n_candidates": len(crop_texts),
        }
        for _, row in df_bench.iterrows()
    ]
    predictions = pd.DataFrame(prediction_rows_with_targets(
        sim, target_idx, crop_texts, row_metadata
    ))
    return overall, pd.DataFrame(rows), predictions


crop_overall_rows = []
crop_per_crop_frames = []
crop_prediction_frames = []
for model_id, payload in embeddings.items():
    overall, per_crop, predictions = evaluate_image_to_crop(model_id, payload)
    crop_overall_rows.append(overall)
    crop_per_crop_frames.append(per_crop)
    crop_prediction_frames.append(predictions)

df_crop_overall = pd.DataFrame(crop_overall_rows)
df_crop_per_crop = pd.concat(crop_per_crop_frames, ignore_index=True)

chance = random_baseline_metrics(len(crop_texts))

In [26]:
print(f"Random chance over {len(crop_texts)} crops: "
      f"R@1={chance['R@1']:.3f},R@3={chance['R@3']:.3f}, MRR={chance['MRR']:.3f}\n")
display(df_crop_overall[["model_id", "N"] + METRIC_COLS].sort_values("R@1", ascending=False))

# --- macro average (unweighted mean over crops) ---------------------------
# Micro (above) is dominated by citrus. Macro gives each crop equal say and is the honest
# headline for an imbalanced task.
macro_rows = []
for model_id, grp in df_crop_per_crop.groupby("model_id"):
    row = {"model_id": model_id, "N": int(grp["N"].sum()), "n_crops": len(grp)}
    for col in METRIC_COLS:
        row[col] = float(grp[col].mean())
    macro_rows.append(row)
df_crop_macro = pd.DataFrame(macro_rows)

print("\n=== Task 1 — macro (unweighted mean over crops) ===")
display(df_crop_macro[["model_id", "N", "n_crops"] + METRIC_COLS]
        .sort_values("R@1", ascending=False))

Random chance over 32 crops: R@1=0.031,R@3=0.094, MRR=0.127



,model_id,N,R@1,R@3,MRR
0,00_siglip2_out_of_box,94,0.627660,0.851064,0.747527
1,26_zs_siglip2_trunc,94,0.627660,0.851064,0.747527
2,26_zs_siglip2_meanpool,94,0.627660,0.851064,0.747527
11,29_sweep_size_5000,94,0.553191,0.723404,0.661611
23,27_last_0,94,0.340426,0.617021,0.517477
26,27_last_6,94,0.297872,0.595745,0.483410
24,27_last_2,94,0.234043,0.521277,0.417374
6,zz_clip_raw,94,0.191489,0.393617,0.354254
3,26_zs_clip_meanpool,94,0.191489,0.393617,0.354254
27,27_last_8,94,0.180851,0.468085,0.377909



=== Task 1 — macro (unweighted mean over crops) ===


,model_id,N,n_crops,R@1,R@3,MRR
0,00_siglip2_out_of_box,94,3,0.426113,0.942387,0.660317
2,26_zs_siglip2_meanpool,94,3,0.426113,0.942387,0.660317
3,26_zs_siglip2_trunc,94,3,0.426113,0.942387,0.660317
1,26_zs_clip_meanpool,94,3,0.419940,0.550505,0.535956
34,zz_clip_raw,94,3,0.419940,0.550505,0.535956
11,27D_last_4,94,3,0.326412,0.452862,0.440226
9,27D_last_0,94,3,0.221661,0.452862,0.419578
25,29_sweep_size_5000,94,3,0.213992,0.306023,0.296752
6,27C_last_4,94,3,0.145903,0.598765,0.409464
24,29_sweep_size_500,94,3,0.140292,0.272353,0.259320


## Section 8 — Task 2: within-crop image → stage

Each photo is ranked **only against the stage descriptions of its own crop**, so this is a within-crop problem: a citrus photo is never compared with a flax stage.

Two levels, evaluated independently:

- **principal** — BBCH crops only. Principal is a BBCH concept; the TI crops use the Smith-Froment letter scale, and the BBCH principal ids derived on TI rows do not exist in their pool, so they are scored on secondary only.
- **secondary** — every crop (2-digit BBCH for citrus, Terres Inovia code for flax/rape).

Both are reported separately for **French** and **English** descriptions.

The overall figure is a **micro-average**: each `(crop, level, language)` cell contributes its own photos, weighted by `N`, so a crop with 87 photos counts more than one with 2.

In [27]:
# Which levels are evaluated per crop kind. Principal is a BBCH concept: TI crops use the
# Smith-Froment letter scale, and the BBCH principal ids derived on TI rows are not in their
# pool, so TI crops are scored on the secondary level only.
EVALUATED_LEVELS = {"bbch": ("principal", "secondary"), "ti": ("secondary",)}
LANGUAGES = ("fr", "en")

# Crop kind per crop, taken from the benchmark rows themselves (notebook 31's CROP_CONFIG
# is not available here). A crop is "bbch" when its rows carry a BBCH secondary id.
CROP_KIND = {}
for crop, grp in df_bench.groupby("crop_name"):
    has_bbch = grp["bbch_secondary_stage_id"].astype(str).str.strip().ne("").any()
    CROP_KIND[crop] = "bbch" if has_bbch else "ti"

print("Crop kind:", CROP_KIND)
for crop, kind in sorted(CROP_KIND.items()):
    print(f"  {crop}: {kind} -> levels {EVALUATED_LEVELS[kind]}")


def evaluate_within_crop_stage(model_id, payload):
    """Rank each photo against its own crop's stage candidates.

    Returns
    -------
    per_crop    : DataFrame  one row per (crop, level, lang)
    overall     : DataFrame  one row per (level, lang), micro-averaged over crops
    predictions : DataFrame  one row per evaluated photo and (level, lang)
    """
    image_emb = payload["image_emb"]
    multi_vector = payload["multi_vector_text"]
    stage_emb = payload["stage_emb"]

    per_crop_rows = []
    prediction_rows = []

    for crop, kind in sorted(CROP_KIND.items()):
        crop_idx = df_bench.index[df_bench["crop_name"] == crop].tolist()
        if not crop_idx:
            continue
        crop_images = image_emb[crop_idx]

        for level in EVALUATED_LEVELS[kind]:
            for lang in LANGUAGES:
                key = (crop, level, lang)
                if key not in stage_emb:
                    continue

                # candidate pool = this crop's stages at this level, in this language
                pool_rows = stage_lists.get((crop, level), [])
                if not pool_rows:
                    continue
                stage_ids = [r["stage_id"] for r in pool_rows]
                id_to_col = {sid: j for j, sid in enumerate(stage_ids)}

                # ground-truth column per photo; skip photos whose id is not in the pool
                gt_col = "gt_principal_id" if level == "principal" else "gt_secondary_id"
                keep_rows, target_cols = [], []
                for pos, row_i in enumerate(crop_idx):
                    gid = str(df_bench.at[row_i, gt_col] or "").strip()
                    if gid in id_to_col:
                        keep_rows.append(pos)
                        target_cols.append(id_to_col[gid])
                if not keep_rows:
                    continue

                sim = similarity_matrix(crop_images[keep_rows], stage_emb[key], multi_vector)
                target_idx = torch.tensor(target_cols, dtype=torch.long)
                m = rank_metrics_with_targets(sim, target_idx)

                per_crop_rows.append({
                    "model_id": model_id,
                    "crop_name": crop,
                    "level": level,
                    "lang": lang,
                    "n_candidates": len(stage_ids),
                    **m,
                })

                row_metadata = []
                for pos in keep_rows:
                    row_i = crop_idx[pos]
                    row_metadata.append({
                        "model_id": model_id,
                        "task": "image->stage",
                        "level": level,
                        "lang": lang,
                        "photo_name": df_bench.at[row_i, "photo_name"],
                        "crop_name": crop,
                        "n_candidates": len(stage_ids),
                    })
                prediction_rows.extend(prediction_rows_with_targets(
                    sim, target_idx, stage_ids, row_metadata
                ))

    per_crop = pd.DataFrame(per_crop_rows)
    predictions = pd.DataFrame(prediction_rows)
    if per_crop.empty:
        return per_crop, pd.DataFrame(), predictions

    # micro-average: weight each (crop, level, lang) cell by its number of photos
    overall_rows = []
    for (level, lang), grp in per_crop.groupby(["level", "lang"]):
        w = grp["N"].to_numpy(dtype=float)
        row = {"model_id": model_id, "level": level, "lang": lang,
               "N": int(grp["N"].sum()), "n_crops": len(grp)}
        for col in METRIC_COLS:
            row[col] = float((grp[col].to_numpy(dtype=float) * w).sum() / w.sum())
        overall_rows.append(row)

    return per_crop, pd.DataFrame(overall_rows), predictions


print("evaluate_within_crop_stage() ready.")

Crop kind: {'citrus': 'bbch', 'oilseedFlax': 'bbch', 'oilseedRape': 'bbch'}
  citrus: bbch -> levels ('principal', 'secondary')
  oilseedFlax: bbch -> levels ('principal', 'secondary')
  oilseedRape: bbch -> levels ('principal', 'secondary')
evaluate_within_crop_stage() ready.


In [28]:
# --- run Task 2 for every model -------------------------------------------
stage_per_crop_frames = []
stage_overall_frames = []
stage_prediction_frames = []

for model_id, payload in embeddings.items():
    per_crop, overall, predictions = evaluate_within_crop_stage(model_id, payload)
    if per_crop.empty:
        print(f"[{model_id}] WARNING: no within-crop stage results")
        continue
    stage_per_crop_frames.append(per_crop)
    stage_overall_frames.append(overall)
    stage_prediction_frames.append(predictions)

df_stage_per_crop = (
    pd.concat(stage_per_crop_frames, ignore_index=True)
    if stage_per_crop_frames else pd.DataFrame()
)
df_stage_overall = (
    pd.concat(stage_overall_frames, ignore_index=True)
    if stage_overall_frames else pd.DataFrame()
)

print(f"Per-crop rows: {len(df_stage_per_crop)}")
print(f"Overall rows:  {len(df_stage_overall)}")

# --- chance levels ---------------------------------------------------------
# Chance depends on the pool size, which differs per (crop, level), so it is computed
# per cell and then micro-averaged the same way as the metrics.
chance_rows = []
for _, r in df_stage_per_crop.iterrows():
    c = random_baseline_metrics(int(r["n_candidates"]))
    chance_rows.append({"crop_name": r["crop_name"], "level": r["level"],
                        "lang": r["lang"], "N": r["N"], **c})
df_stage_chance = pd.DataFrame(chance_rows)

chance_overall_rows = []
for (level, lang), grp in df_stage_chance.groupby(["level", "lang"]):
    w = grp["N"].to_numpy(dtype=float)
    row = {"level": level, "lang": lang, "N": int(grp["N"].sum())}
    for col in METRIC_COLS:
        row[col] = float((grp[col].to_numpy(dtype=float) * w).sum() / w.sum())
    chance_overall_rows.append(row)
df_stage_chance_overall = pd.DataFrame(chance_overall_rows)

print("\n=== Task 2 — overall (micro-averaged over crops) ===")
display(df_stage_overall[["model_id", "level", "lang", "N", "n_crops"] + METRIC_COLS]
        .sort_values(["level", "lang", "model_id"]))

print("\n=== Task 2 — chance level ===")
display(df_stage_chance_overall[["level", "lang", "N"] + METRIC_COLS])

Per-crop rows: 370
Overall rows:  148

=== Task 2 — overall (micro-averaged over crops) ===


,model_id,level,lang,N,n_crops,R@1,R@3,MRR
0,00_siglip2_out_of_box,principal,en,92,2,0.304348,0.532609,0.472511
12,26_zs_clip_meanpool,principal,en,92,2,0.260870,0.576087,0.464596
8,26_zs_siglip2_meanpool,principal,en,92,2,0.304348,0.532609,0.472511
4,26_zs_siglip2_trunc,principal,en,92,2,0.304348,0.532609,0.472511
112,27C_last_0,principal,en,92,2,0.206522,0.413043,0.394626
...,...,...,...,...,...,...,...,...
67,smaller_size_500,secondary,fr,91,3,0.054945,0.098901,0.151491
79,smaller_size_5000,secondary,fr,91,3,0.054945,0.197802,0.190002
27,zz_clip_raw,secondary,fr,91,3,0.032967,0.120879,0.144992
23,zz_dinov3_bge_raw,secondary,fr,91,3,0.000000,0.065934,0.084867



=== Task 2 — chance level ===


,level,lang,N,R@1,R@3,MRR
0,principal,en,629,0.117647,0.352941,0.326284
1,principal,fr,629,0.117647,0.352941,0.326284
2,secondary,en,5328,0.020833,0.062500,0.090637
3,secondary,fr,5328,0.020833,0.062500,0.090637


## Section 9 — Persist results and compare models

Writes, per `notebook-conventions`:

| file | content |
|---|---|
| `benchmark/bbch_results_<model_id>.csv` | that model's Task 1 + Task 2 rows (overall and per crop) |
| `benchmark/bbch_results_comparison.csv` | every model, every task/level/language, with chance levels |
| `results/<model_id>_predictions.csv` | that model's per-photo predictions for every evaluated task/level/language |

Prediction files contain one row per evaluated photo and task variant, with `gt_id`, top-1 `predicted_id`, tie-aware `gt_rank`, `is_correct`, similarities, and the top-5 candidate ids/scores (JSON-encoded lists). Rows skipped by the evaluation (for example, a stage ground truth outside its candidate pool) are also skipped here.

Metrics schema: `model_id, task, level, lang, crop_name, N, n_candidates, R@1, R@3, MRR` — one row per evaluated unit, columns in a fixed order so the files are diff-able. Plain UTF-8, `index=False`.

In [29]:
# --- assemble one long-format results frame --------------------------------
RESULT_COLS = ["model_id", "task", "level", "lang", "crop_name",
               "N", "n_candidates"] + METRIC_COLS


def _frame(df, task, level=None, lang=None, crop_name=None, n_candidates=None):
    """Normalise a results frame to RESULT_COLS, filling the missing identifiers."""
    out = df.copy()
    out["task"] = task
    if level is not None:
        out["level"] = level
    if lang is not None:
        out["lang"] = lang
    if crop_name is not None:
        out["crop_name"] = crop_name
    if n_candidates is not None:
        out["n_candidates"] = n_candidates
    for col in RESULT_COLS:
        if col not in out.columns:
            out[col] = pd.NA
    return out[RESULT_COLS]


result_frames = []

# Task 1 — image -> crop (overall + per crop)
result_frames.append(_frame(df_crop_overall, "image->crop",
                            level="crop", lang="", crop_name="ALL",
                            n_candidates=len(crop_texts)))
result_frames.append(_frame(df_crop_per_crop, "image->crop",
                            level="crop", lang="",
                            n_candidates=len(crop_texts)))

# Task 2 — within-crop image -> stage (overall + per crop)
if not df_stage_overall.empty:
    result_frames.append(_frame(df_stage_overall, "image->stage", crop_name="ALL"))
    result_frames.append(_frame(df_stage_per_crop, "image->stage"))

df_results = pd.concat(result_frames, ignore_index=True)
df_results = df_results.sort_values(
    ["task", "level", "lang", "crop_name", "model_id"]
).reset_index(drop=True)

# --- per-model CSVs --------------------------------------------------------
for model_id, grp in df_results.groupby("model_id"):
    path = BENCH_DIR / f"bbch_results_{model_id}.csv"
    grp.to_csv(path, index=False, encoding="utf-8")
    print(f"Wrote {path} ({len(grp)} rows)")

# --- per-model, per-photo prediction CSVs ----------------------------------
PREDICTION_COLS = [
    "model_id", "task", "level", "lang", "photo_name", "crop_name",
    "n_candidates", "gt_id", "predicted_id", "gt_rank", "is_correct",
    "gt_similarity", "predicted_similarity", "top_candidate_ids",
    "top_candidate_scores",
]
prediction_frames = crop_prediction_frames + stage_prediction_frames
df_predictions = pd.concat(prediction_frames, ignore_index=True)[PREDICTION_COLS]
df_predictions = df_predictions.sort_values(
    ["model_id", "task", "level", "lang", "photo_name"]
).reset_index(drop=True)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for model_id, grp in df_predictions.groupby("model_id", sort=True):
    path = RESULTS_DIR / f"{model_id}_predictions.csv"
    grp.to_csv(path, index=False, encoding="utf-8")
    print(f"Wrote {path} ({len(grp)} prediction rows)")

# --- cross-model comparison (overall rows only) ----------------------------
df_comparison = df_results[df_results["crop_name"] == "ALL"].copy()

# chance level per (task, level, lang)
chance_lookup = {("image->crop", "crop", ""): chance}
for _, r in df_stage_chance_overall.iterrows():
    chance_lookup[("image->stage", r["level"], r["lang"])] = r.to_dict()

chance_rows = []
for (task, level, lang), grp in df_comparison.groupby(["task", "level", "lang"]):
    c = chance_lookup.get((task, level, lang))
    if c is None:
        continue
    chance_rows.append({"model_id": "CHANCE", "task": task, "level": level, "lang": lang,
                        "crop_name": "ALL", "N": int(grp["N"].iloc[0]),
                        "n_candidates": grp["n_candidates"].iloc[0],
                        **{col: c[col] for col in METRIC_COLS}})

df_comparison = pd.concat(
    [df_comparison, pd.DataFrame(chance_rows)[RESULT_COLS]], ignore_index=True
).sort_values(["task", "level", "lang", "R@1"], ascending=[True, True, True, False])

comparison_path = BENCH_DIR / "bbch_results_comparison.csv"
df_comparison.to_csv(comparison_path, index=False, encoding="utf-8")
print(f"\nWrote {comparison_path} ({len(df_comparison)} rows)")

print("\n=== Cross-model comparison (overall, with chance) ===")
display(df_comparison[["task", "level", "lang", "model_id", "N", "n_candidates"] + METRIC_COLS]
        .reset_index(drop=True))

Wrote /projects/agrifutur/sji/bsv-crossmodalIR/BBCH case study/benchmark/bbch_results_00_siglip2_out_of_box.csv (18 rows)
Wrote /projects/agrifutur/sji/bsv-crossmodalIR/BBCH case study/benchmark/bbch_results_26_zs_clip_meanpool.csv (18 rows)
Wrote /projects/agrifutur/sji/bsv-crossmodalIR/BBCH case study/benchmark/bbch_results_26_zs_siglip2_meanpool.csv (18 rows)
Wrote /projects/agrifutur/sji/bsv-crossmodalIR/BBCH case study/benchmark/bbch_results_26_zs_siglip2_trunc.csv (18 rows)
Wrote /projects/agrifutur/sji/bsv-crossmodalIR/BBCH case study/benchmark/bbch_results_27C_last_0.csv (18 rows)
Wrote /projects/agrifutur/sji/bsv-crossmodalIR/BBCH case study/benchmark/bbch_results_27C_last_2.csv (18 rows)
Wrote /projects/agrifutur/sji/bsv-crossmodalIR/BBCH case study/benchmark/bbch_results_27C_last_4.csv (18 rows)
Wrote /projects/agrifutur/sji/bsv-crossmodalIR/BBCH case study/benchmark/bbch_results_27C_last_6.csv (18 rows)
Wrote /projects/agrifutur/sji/bsv-crossmodalIR/BBCH case study/benchmar

,task,level,lang,model_id,N,n_candidates,R@1,R@3,MRR
0,image->crop,crop,,00_siglip2_out_of_box,94,32,0.627660,0.851064,0.747527
1,image->crop,crop,,26_zs_siglip2_meanpool,94,32,0.627660,0.851064,0.747527
2,image->crop,crop,,26_zs_siglip2_trunc,94,32,0.627660,0.851064,0.747527
3,image->crop,crop,,29_sweep_size_5000,94,32,0.553191,0.723404,0.661611
4,image->crop,crop,,27_last_0,94,32,0.340426,0.617021,0.517477
...,...,...,...,...,...,...,...,...,...
185,image->stage,secondary,fr,CHANCE,91,<NA>,0.020833,0.062500,0.090637
186,image->stage,secondary,fr,27C_last_4,91,<NA>,0.010989,0.120879,0.127340
187,image->stage,secondary,fr,27C_last_8,91,<NA>,0.010989,0.142857,0.138260
188,image->stage,secondary,fr,27_last_4,91,<NA>,0.010989,0.142857,0.147327
